In [ ]:
pip install powerlaw networkx numpy scipy pandas tqdm scipy

In [1]:
import sys
import logging
from pathlib import Path

# sys.path manipulation is required because ysocial_validator is not installed as a package
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# scripts/ must be on sys.path for dynamic imports of numbered scripts (e.g., importlib.import_module("01_..."))
scripts_path = str(Path("../scripts").resolve())
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s")

from ysocial_validator.ingestion import YSocialGraphBuilder

db_path = Path("../data/00_raw/benchmark_runs/run01.sqlite").resolve()

G = YSocialGraphBuilder(str(db_path)).load_follower_graph()

numexpr.utils — Note: NumExpr detected 12 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
numexpr.utils — NumExpr defaulting to 8 threads.
ysocial_validator.ingestion — Graph constructed — nodes: 1819, edges: 964, end_round=None


In [2]:
import pandas as pd
from ysocial_validator.topometrics import YSocialTopometrics

report = YSocialTopometrics(G).generate_full_report()

pd.Series(report).rename("value").to_frame()

,value
num_nodes,1819
num_edges,964
density,0.000292
avg_in_degree,0.529962
std_in_degree,1.002229
avg_out_degree,0.529962
std_out_degree,4.063076
reciprocity,0.006224
alpha_in_degree,2.3066
xmin,1.0


In [3]:
from importlib import import_module

stage_a_validator = import_module("01_stage_a_validator")
StageAValidator = stage_a_validator.StageAValidator

data_dir = Path("../data/00_raw/benchmark_runs").resolve()

validator = StageAValidator(str(data_dir))
raw_df = validator.process_runs()

output_csv = Path("../data/01_processed/stage_a_raw.csv").resolve()
validator.save_raw_results(str(output_csv))

report = validator.full_stability_report()
print("\n=== Topological Stability (Stage A) ===\n")
print(report["stability"])

01_stage_a_validator — StageAValidator initialized — 30 databases found in 'benchmark_runs'
Processing Stage A runs:   0%|          | 0/30 [00:00<?, ?run/s]ysocial_validator.ingestion — Graph constructed — nodes: 1819, edges: 964, end_round=None
01_stage_a_validator — Run 'run01' completed — nodes: 1819, edges: 964
Processing Stage A runs:   3%|▎         | 1/30 [00:00<00:10,  2.80run/s]ysocial_validator.ingestion — Graph constructed — nodes: 1850, edges: 986, end_round=None
01_stage_a_validator — Run 'run02' completed — nodes: 1850, edges: 986
Processing Stage A runs:   7%|▋         | 2/30 [00:00<00:10,  2.79run/s]ysocial_validator.ingestion — Graph constructed — nodes: 1736, edges: 910, end_round=None
01_stage_a_validator — Run 'run03' completed — nodes: 1736, edges: 910
Processing Stage A runs:  10%|█         | 3/30 [00:01<00:08,  3.06run/s]ysocial_validator.ingestion — Graph constructed — nodes: 1906, edges: 999, end_round=None
01_stage_a_validator — Run 'run04' completed — nodes: 1


=== Topological Stability (Stage A) ===

                     n      mean       std       sem  ci95_low  ci95_high  \
metric                                                                      
alpha_in_degree     30  2.359300  0.048945  0.008936  2.341024   2.377576   
modularity          30  0.608153  0.016620  0.003034  0.601947   0.614359   
average_clustering  30  0.013227  0.003230  0.000590  0.012020   0.014433   
density             30  0.000287  0.000021  0.000004  0.000279   0.000295   

                        cv  
metric                      
alpha_in_degree     0.0207  
modularity          0.0273  
average_clustering  0.2442  
density             0.0729  


In [4]:
from importlib import import_module

stage_a_viz = import_module("02_stage_a_visualization")
StageAVisualizer = stage_a_viz.StageAVisualizer

viz = StageAVisualizer("../data/01_processed/stage_a_raw.csv")
viz.plot_stability_distributions("../data/01_processed/stage_a_stability.png")

# Optional: print text summary to stdout
print(viz.plot_summary_statistics())

02_stage_a_visualization — StageAVisualizer initialized — CSV loaded with 30 rows.
D:\University\Social Network Analysis\YSocial-Topology-Validator\scripts\02_stage_a_visualization.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
matplotlib.font_manager — Fontsize 0.00 < 1.0 pt not allowed by FreeType. Setting fontsize = 1 pt
D:\University\Social Network Analysis\YSocial-Topology-Validator\scripts\02_stage_a_visualization.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
matplotlib.font_manager — Fontsize 0.00 < 1.0 pt not allowed by FreeType. Setting fontsize = 1 pt
D:\University\Social Network Analysis\YSocial-Topology-Validator\scripts\02_stage_a_visualization.py:96: FutureWarn

=== Stability Report (Stage A) ===

Power-law Exponent (α)........ mean=2.3593, std=0.0489, CV=0.0207
Modularity (Q)................ mean=0.6082, std=0.0166, CV=0.0273
Average Clustering............ mean=0.0132, std=0.0032, CV=0.2442
Network Density............... mean=0.0003, std=0.0000, CV=0.0729



In [5]:
from importlib import import_module

stage_b_analyzer_mod = import_module("03_stage_b_analyzer")
StageBAnalyzer = stage_b_analyzer_mod.StageBAnalyzer

analyzer = StageBAnalyzer("../data/00_raw/sensitivity_runs/")
raw_df = analyzer.process_all_runs()
analyzer.save_raw_results("../data/01_processed/stage_b_raw.csv")
analyzer.save_aggregated_results("../data/01_processed/stage_b_aggregated.csv")

report = analyzer.full_sensitivity_report()
print(report["aggregated"])  # 11 rows, one per condition

03_stage_b_analyzer — StageBAnalyzer initialized — 11 conditions found in 'sensitivity_runs'
Processing Stage B runs:   0%|          | 0/110 [00:00<?, ?run/s]ysocial_validator.ingestion — Graph constructed — nodes: 99, edges: 18, end_round=None
c:\Users\Fabrizio\anaconda3\Lib\site-packages\powerlaw\fitting.py:412: UserWarning: Less than 2 unique data values for fitting xmin! Returning nans.
  warnings.warn("Less than 2 unique data values for fitting xmin! Returning nans.")
ysocial_validator.ingestion — Graph constructed — nodes: 100, edges: 23, end_round=None
c:\Users\Fabrizio\anaconda3\Lib\site-packages\powerlaw\fitting.py:412: UserWarning: Less than 2 unique data values for fitting xmin! Returning nans.
  warnings.warn("Less than 2 unique data values for fitting xmin! Returning nans.")
ysocial_validator.ingestion — Graph constructed — nodes: 103, edges: 29, end_round=None
c:\Users\Fabrizio\anaconda3\Lib\site-packages\powerlaw\fitting.py:412: UserWarning: Less than 2 unique data value

   condition  modularity_mean  modularity_std  modularity_ci95_low  \
0         c0          0.64826        0.078156             0.592350   
1         c1          0.49473        0.136959             0.396756   
2        c10          0.64743        0.084523             0.586966   
3         c2          0.64505        0.075160             0.591284   
4         c3          0.59936        0.085797             0.537984   
5         c4          0.63588        0.041501             0.606192   
6         c5          0.66121        0.047697             0.627090   
7         c6          0.62953        0.074899             0.575951   
8         c7          0.63274        0.058086             0.591188   
9         c8          0.66275        0.069049             0.613355   
10        c9          0.57009        0.091812             0.504412   

    modularity_ci95_high  average_clustering_mean  average_clustering_std  \
0               0.704170                  0.00392                0.007231   
1    

In [6]:
from importlib import import_module

stage_b_viz = import_module("04_stage_b_visualization")
StageBVisualizer = stage_b_viz.StageBVisualizer

viz = StageBVisualizer("../data/01_processed/stage_b_raw.csv")
viz.plot_modularity_comparison("../data/01_processed/stage_b_modularity_comparison.png")

# Optional: tabular modularity summary by condition
print(viz.get_summary_statistics())

04_stage_b_visualization — StageBVisualizer initialized — CSV loaded with 110 rows.
D:\University\Social Network Analysis\YSocial-Topology-Validator\scripts\04_stage_b_visualization.py:112: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
04_stage_b_visualization — Figure saved — file: 'D:\University\Social Network Analysis\YSocial-Topology-Validator\data\01_processed\stage_b_modularity_comparison.png'


  condition              label   n     mean       std       sem
0        c0           Baseline  10  0.64826  0.078156  0.024715
1        c1    Neutral Persona  10  0.49473  0.136959  0.043310
2        c3    Low Temperature  10  0.59936  0.085797  0.027131
3        c4   High Temperature  10  0.63588  0.041501  0.013124
4        c8  Aggressive RecSys  10  0.66275  0.069049  0.021835


In [7]:
from importlib import import_module

stage_b_hyp = import_module("05_stage_b_hypothesis")
StageBHypothesisTesting = stage_b_hyp.StageBHypothesisTesting

tester = StageBHypothesisTesting("../data/01_processed/stage_b_raw.csv")
results_df = tester.run_tests()

tester.print_results(verbose=True)

tester.save_results("../data/01_processed/stage_b_pvalues.csv")

# Conditions with p < 0.05
sig_conditions = tester.get_significant_conditions(alpha=0.05)
print(f"Significant conditions (p<0.05): {sig_conditions}")

05_stage_b_hypothesis — StageBHypothesisTesting initialized — 110 rows loaded.
05_stage_b_hypothesis — c0 vs c1: Δμ=-0.1535, U=86.00, p=0.007285 **
05_stage_b_hypothesis — c0 vs c3: Δμ=-0.0489, U=68.00, p=0.185877 ns
05_stage_b_hypothesis — c0 vs c4: Δμ=-0.0124, U=51.00, p=0.969839 ns
05_stage_b_hypothesis — c0 vs c8: Δμ=0.0145, U=40.00, p=0.472676 ns
05_stage_b_hypothesis — Test results saved — file: 'D:\University\Social Network Analysis\YSocial-Topology-Validator\data\01_processed\stage_b_pvalues.csv', rows: 4



MANN-WHITNEY U TEST RESULTS (Baseline: c0, Modularity)
Condition             Label  Mean_Baseline  Mean_Test  Mean_Difference  U_statistic  p_value Significance
       c1   Neutral Persona        0.64826    0.49473         -0.15353         86.0 0.007285           **
       c3   Low Temperature        0.64826    0.59936         -0.04890         68.0 0.185877           ns
       c4  High Temperature        0.64826    0.63588         -0.01238         51.0 0.969839           ns
       c8 Aggressive RecSys        0.64826    0.66275          0.01449         40.0 0.472676           ns
Significance: *** p<0.001 (highly significant), ** p<0.01 (very significant), * p<0.05 (significant), ns (not significant)

Significant conditions (p<0.05): ['c1']
